In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json

# Find the project root automatically
CURRENT_DIR = Path.cwd()

PROJECT_ROOT = None

for folder in [CURRENT_DIR, *CURRENT_DIR.parents]:
    if (folder / "data" / "processed").is_dir() and (folder / "Raw_datasets").is_dir():
        PROJECT_ROOT = folder
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Project folders not found. Open this notebook inside your project "
        "and check that data/processed and Raw_datasets exist."
    )

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
OUTPUTS = PROJECT_ROOT / "outputs"
REPORTS = OUTPUTS / "evaluation_reports"

REPORTS.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Processed data folder:", DATA_PROCESSED)
print("Reports folder:", REPORTS)

Project root: d:\Dynamic_AI_Chatbot_Project
Processed data folder: d:\Dynamic_AI_Chatbot_Project\data\processed
Reports folder: d:\Dynamic_AI_Chatbot_Project\outputs\evaluation_reports


In [2]:
# Load the three cleaned datasets

telecom_excel_path = DATA_PROCESSED / "telecom_churn_cleaned.xlsx"
banking_path = DATA_PROCESSED / "banking_churn_cleaned.csv"
telecom_csv_path = DATA_PROCESSED / "telecom_churn_second_cleaned.csv"

# Check that all files exist
for file_path in [telecom_excel_path, banking_path, telecom_csv_path]:
    print(file_path.name, "->", file_path.exists())

if not all(p.exists() for p in [
    telecom_excel_path, banking_path, telecom_csv_path
]):
    raise FileNotFoundError(
        "One or more cleaned datasets are missing. Check data/processed."
    )

# Load datasets separately
telecom_excel = pd.read_excel(telecom_excel_path, engine="openpyxl")
banking = pd.read_csv(banking_path)
telecom_csv = pd.read_csv(telecom_csv_path)

print("\nAll datasets loaded successfully.")
print("Telecom Excel shape:", telecom_excel.shape)
print("Banking shape:", banking.shape)
print("Second Telecom CSV shape:", telecom_csv.shape)

telecom_churn_cleaned.xlsx -> True
banking_churn_cleaned.csv -> True
telecom_churn_second_cleaned.csv -> True

All datasets loaded successfully.
Telecom Excel shape: (7043, 33)
Banking shape: (10000, 17)
Second Telecom CSV shape: (7043, 21)


In [3]:
# Inspect column names and possible target columns

datasets = {
    "Telecom Excel": telecom_excel,
    "Banking": banking,
    "Second Telecom CSV": telecom_csv
}

for name, df in datasets.items():
    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)
    print("Columns:")
    print(df.columns.tolist())
    print("\nFirst 3 rows:")
    display(df.head(3))


Telecom Excel
Columns:
['customerid', 'count', 'country', 'state', 'city', 'zip_code', 'lat_long', 'latitude', 'longitude', 'gender', 'senior_citizen', 'partner', 'dependents', 'tenure_months', 'phone_service', 'multiple_lines', 'internet_service', 'online_security', 'online_backup', 'device_protection', 'tech_support', 'streaming_tv', 'streaming_movies', 'contract', 'paperless_billing', 'payment_method', 'monthly_charges', 'total_charges', 'churn_label', 'churn_value', 'churn_score', 'cltv', 'churn_reason']

First 3 rows:


,customerid,count,country,state,city,zip_code,lat_long,latitude,longitude,gender,...,contract,paperless_billing,payment_method,monthly_charges,total_charges,churn_label,churn_value,churn_score,cltv,churn_reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.50,Yes,1,86,5372,Moved



Banking
Columns:
['customerid', 'surname', 'creditscore', 'location', 'gender', 'age', 'tenure', 'account_balance', 'numofproducts', 'hascrcard', 'isactivemember', 'estimatedsalary', 'exited', 'complain', 'satisfaction_score', 'card_type', 'point_earned']

First 3 rows:


,customerid,surname,creditscore,location,gender,age,tenure,account_balance,numofproducts,hascrcard,isactivemember,estimatedsalary,exited,complain,satisfaction_score,card_type,point_earned
0,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1,1,2,DIAMOND,464
1,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,1,3,DIAMOND,456
2,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,1,3,DIAMOND,377



Second Telecom CSV
Columns:
['customerid', 'gender', 'seniorcitizen', 'partner', 'dependents', 'tenure', 'phoneservice', 'multiplelines', 'internetservice', 'onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport', 'streamingtv', 'streamingmovies', 'contract', 'paperlessbilling', 'paymentmethod', 'monthlycharges', 'totalcharges', 'churn']

First 3 rows:


,customerid,gender,seniorcitizen,partner,dependents,tenure,phoneservice,multiplelines,internetservice,onlinesecurity,...,deviceprotection,techsupport,streamingtv,streamingmovies,contract,paperlessbilling,paymentmethod,monthlycharges,totalcharges,churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes


In [4]:
# Identify possible churn target columns

target_keywords = [
    "churn",
    "exited",
    "attrition",
    "churn_value",
    "churn_label",
    "attrition_flag"
]

for name, df in datasets.items():
    possible_targets = [
        col for col in df.columns
        if any(keyword in col.lower().replace(" ", "_")
               for keyword in target_keywords)
    ]

    print(f"\n{name}")
    print("Possible target columns:", possible_targets)

    for col in possible_targets:
        print(f"\nValues in {col}:")
        print(df[col].value_counts(dropna=False).head(15))


Telecom Excel
Possible target columns: ['churn_label', 'churn_value', 'churn_score', 'churn_reason']

Values in churn_label:
churn_label
No     5174
Yes    1869
Name: count, dtype: int64

Values in churn_value:
churn_value
0    5174
1    1869
Name: count, dtype: int64

Values in churn_score:
churn_score
80    151
71    148
77    145
67    143
76    141
68    141
70    140
69    140
78    138
72    137
65    136
66    135
75    129
74    128
73    126
Name: count, dtype: int64

Values in churn_reason:
churn_reason
NaN                                          5174
Attitude of support person                    192
Competitor offered higher download speeds     189
Competitor offered more data                  162
Don't know                                    154
Competitor made better offer                  140
Attitude of service provider                  135
Competitor had better devices                 130
Network reliability                           103
Product dissatisfaction       

In [5]:
# Customer ID and Data Leakage Checks.

id_keywords = [
    "customerid",
    "customer_id",
    "customer id",
    "clientid",
    "client_id",
    "accountid",
    "account_id"
]

for name, df in datasets.items():
    possible_ids = [
        col for col in df.columns
        if col.lower().replace(" ", "").replace("-", "_")
        in id_keywords
    ]

    print(f"\n{name}")
    print("Possible ID columns:", possible_ids)


Telecom Excel
Possible ID columns: ['customerid']

Banking
Possible ID columns: ['customerid']

Second Telecom CSV
Possible ID columns: ['customerid']


In [6]:
# Update these two names only if Cell 5 identifies different ID columns.
TELECOM_EXCEL_ID = "CustomerID"
TELECOM_CSV_ID = "CustomerID"

if TELECOM_EXCEL_ID in telecom_excel.columns and TELECOM_CSV_ID in telecom_csv.columns:

    excel_ids = set(
        telecom_excel[TELECOM_EXCEL_ID].dropna().astype(str).str.strip()
    )

    csv_ids = set(
        telecom_csv[TELECOM_CSV_ID].dropna().astype(str).str.strip()
    )

    overlapping_ids = excel_ids.intersection(csv_ids)

    print("Unique IDs in Telecom Excel:", len(excel_ids))
    print("Unique IDs in Telecom CSV:", len(csv_ids))
    print("Overlapping customer IDs:", len(overlapping_ids))

else:
    print("ID column names need to be updated using Cell 5 output.")

ID column names need to be updated using Cell 5 output.


In [7]:
for name, df in datasets.items():
    print("\n" + "=" * 55)
    print(name)
    print("=" * 55)

    print("Duplicate rows:", df.duplicated().sum())
    print("\nMissing values:")
    display(
        df.isnull().sum()
        .sort_values(ascending=False)
        .to_frame("Missing Count")
    )


Telecom Excel
Duplicate rows: 0

Missing values:


,Missing Count
churn_reason,5174
total_charges,11
customerid,0
count,0
country,0
zip_code,0
lat_long,0
state,0
city,0
gender,0



Banking
Duplicate rows: 0

Missing values:


,Missing Count
customerid,0
surname,0
creditscore,0
location,0
gender,0
age,0
tenure,0
account_balance,0
numofproducts,0
hascrcard,0



Second Telecom CSV
Duplicate rows: 0

Missing values:


,Missing Count
totalcharges,11
gender,0
seniorcitizen,0
partner,0
customerid,0
dependents,0
tenure,0
multiplelines,0
phoneservice,0
onlinesecurity,0


In [8]:
print("BANKING DATASET QUALITY SUMMARY")

print("\nDataset shape:", banking.shape)

print("\nTotal missing values:", banking.isnull().sum().sum())

print("\nColumns with missing values:")
print(banking.isnull().sum()[banking.isnull().sum() > 0])

print("\nDuplicate rows:", banking.duplicated().sum())

print("\nColumn names:")
print(banking.columns.tolist())

BANKING DATASET QUALITY SUMMARY

Dataset shape: (10000, 17)

Total missing values: 0

Columns with missing values:
Series([], dtype: int64)

Duplicate rows: 0

Column names:
['customerid', 'surname', 'creditscore', 'location', 'gender', 'age', 'tenure', 'account_balance', 'numofproducts', 'hascrcard', 'isactivemember', 'estimatedsalary', 'exited', 'complain', 'satisfaction_score', 'card_type', 'point_earned']


In [9]:
# Keep the original cleaned datasets unchanged

telecom_excel_ml = telecom_excel.copy()
telecom_csv_ml = telecom_csv.copy()
banking_ml = banking.copy()

print("Working copies created successfully.")

print("Telecom Excel:", telecom_excel_ml.shape)
print("Telecom CSV:", telecom_csv_ml.shape)
print("Banking:", banking_ml.shape)

Working copies created successfully.
Telecom Excel: (7043, 33)
Telecom CSV: (7043, 21)
Banking: (10000, 17)


In [10]:
def standardize_columns(df):
    df = df.copy()

    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(r"[^a-zA-Z0-9]+", "_", regex=True)
        .str.strip("_")
    )

    return df


telecom_excel_ml = standardize_columns(telecom_excel_ml)
telecom_csv_ml = standardize_columns(telecom_csv_ml)
banking_ml = standardize_columns(banking_ml)

print("Column names standardized.")

for name, df in {
    "Telecom Excel": telecom_excel_ml,
    "Telecom CSV": telecom_csv_ml,
    "Banking": banking_ml
}.items():
    print(f"\n{name}:")
    print(df.columns.tolist())

Column names standardized.

Telecom Excel:
['customerid', 'count', 'country', 'state', 'city', 'zip_code', 'lat_long', 'latitude', 'longitude', 'gender', 'senior_citizen', 'partner', 'dependents', 'tenure_months', 'phone_service', 'multiple_lines', 'internet_service', 'online_security', 'online_backup', 'device_protection', 'tech_support', 'streaming_tv', 'streaming_movies', 'contract', 'paperless_billing', 'payment_method', 'monthly_charges', 'total_charges', 'churn_label', 'churn_value', 'churn_score', 'cltv', 'churn_reason']

Telecom CSV:
['customerid', 'gender', 'seniorcitizen', 'partner', 'dependents', 'tenure', 'phoneservice', 'multiplelines', 'internetservice', 'onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport', 'streamingtv', 'streamingmovies', 'contract', 'paperlessbilling', 'paymentmethod', 'monthlycharges', 'totalcharges', 'churn']

Banking:
['customerid', 'surname', 'creditscore', 'location', 'gender', 'age', 'tenure', 'account_balance', 'numofproducts', 'h

In [11]:
target_keywords = [
    "churn",
    "exited",
    "attrition"
]

for name, df in {
    "Telecom Excel": telecom_excel_ml,
    "Telecom CSV": telecom_csv_ml,
    "Banking": banking_ml
}.items():

    possible_targets = [
        col for col in df.columns
        if any(keyword in col for keyword in target_keywords)
    ]

    print(f"\n{name}")
    print("Possible target columns:", possible_targets)

    for col in possible_targets:
        print(df[col].value_counts(dropna=False))


Telecom Excel
Possible target columns: ['churn_label', 'churn_value', 'churn_score', 'churn_reason']
churn_label
No     5174
Yes    1869
Name: count, dtype: int64
churn_value
0    5174
1    1869
Name: count, dtype: int64
churn_score
80    151
71    148
77    145
67    143
76    141
     ... 
88     33
9       3
8       2
7       2
5       1
Name: count, Length: 85, dtype: int64
churn_reason
NaN                                          5174
Attitude of support person                    192
Competitor offered higher download speeds     189
Competitor offered more data                  162
Don't know                                    154
Competitor made better offer                  140
Attitude of service provider                  135
Competitor had better devices                 130
Network reliability                           103
Product dissatisfaction                       102
Price too high                                 98
Service dissatisfaction                        89
Lack 

In [12]:
# Check whether churn_label and churn_value represent the same outcome

telecom_target_check = (
    telecom_excel_ml["churn_label"].astype(str).str.strip().str.lower()
    == telecom_excel_ml["churn_value"].map({0: "no", 1: "yes"})
)

print("Rows checked:", len(telecom_target_check))
print("Matching target values:", telecom_target_check.sum())
print("Mismatching target values:", (~telecom_target_check).sum())

if telecom_target_check.all():
    print("\nConfirmed: churn_label and churn_value agree.")
else:
    print("\nWARNING: Some target values do not agree. Inspect these rows:")
    display(
        telecom_excel_ml.loc[
            ~telecom_target_check,
            ["churn_label", "churn_value"]
        ].head(20)
    )

Rows checked: 7043
Matching target values: 7043
Mismatching target values: 0

Confirmed: churn_label and churn_value agree.


In [13]:
for name, df in {
    "Telecom Excel": telecom_excel_ml,
    "Telecom CSV": telecom_csv_ml,
    "Banking": banking_ml
}.items():

    print(f"\n{name} — possible ID columns")

    for col in df.columns:
        if "customer" in col and "id" in col:
            print(col)


Telecom Excel — possible ID columns
customerid

Telecom CSV — possible ID columns
customerid

Banking — possible ID columns
customerid


In [14]:
# Update these names if Cell 12 prints different ID columns.
EXCEL_ID = "customer_id"
CSV_ID = "customer_id"

if EXCEL_ID not in telecom_excel_ml.columns:
    print("Excel ID column not found. Update EXCEL_ID.")
elif CSV_ID not in telecom_csv_ml.columns:
    print("CSV ID column not found. Update CSV_ID.")
else:
    excel_ids = set(
        telecom_excel_ml[EXCEL_ID].dropna().astype(str).str.strip()
    )

    csv_ids = set(
        telecom_csv_ml[CSV_ID].dropna().astype(str).str.strip()
    )

    overlap = excel_ids.intersection(csv_ids)

    print("Unique Excel telecom IDs:", len(excel_ids))
    print("Unique CSV telecom IDs:", len(csv_ids))
    print("Overlapping IDs:", len(overlap))

    print("\nSample overlapping IDs:")
    print(list(overlap)[:10])

Excel ID column not found. Update EXCEL_ID.


In [15]:
for name, df in {
    "Telecom Excel": telecom_excel_ml,
    "Telecom CSV": telecom_csv_ml,
    "Banking": banking_ml
}.items():

    print(f"\n{name}")
    print(df.columns.tolist())


Telecom Excel
['customerid', 'count', 'country', 'state', 'city', 'zip_code', 'lat_long', 'latitude', 'longitude', 'gender', 'senior_citizen', 'partner', 'dependents', 'tenure_months', 'phone_service', 'multiple_lines', 'internet_service', 'online_security', 'online_backup', 'device_protection', 'tech_support', 'streaming_tv', 'streaming_movies', 'contract', 'paperless_billing', 'payment_method', 'monthly_charges', 'total_charges', 'churn_label', 'churn_value', 'churn_score', 'cltv', 'churn_reason']

Telecom CSV
['customerid', 'gender', 'seniorcitizen', 'partner', 'dependents', 'tenure', 'phoneservice', 'multiplelines', 'internetservice', 'onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport', 'streamingtv', 'streamingmovies', 'contract', 'paperlessbilling', 'paymentmethod', 'monthlycharges', 'totalcharges', 'churn']

Banking
['customerid', 'surname', 'creditscore', 'location', 'gender', 'age', 'tenure', 'account_balance', 'numofproducts', 'hascrcard', 'isactivemember', '

In [16]:
excel_ids = set(
    telecom_excel_ml["customerid"]
    .dropna()
    .astype(str)
    .str.strip()
)

csv_ids = set(
    telecom_csv_ml["customerid"]
    .dropna()
    .astype(str)
    .str.strip()
)

overlapping_ids = excel_ids.intersection(csv_ids)

print("Telecom Excel unique IDs:", len(excel_ids))
print("Telecom CSV unique IDs:", len(csv_ids))
print("Overlapping customer IDs:", len(overlapping_ids))

print("\nSample overlapping IDs:")
print(list(overlapping_ids)[:10])

Telecom Excel unique IDs: 7043
Telecom CSV unique IDs: 7043
Overlapping customer IDs: 7043

Sample overlapping IDs:
['6051-PTVNS', '2080-CAZNM', '3519-ZKXGG', '2607-FBDFF', '0396-HUJBP', '4828-FAZPK', '0867-LDTTC', '6996-KNSML', '6179-GJPSO', '9167-APMXZ']


In [17]:
print("TELECOM EXCEL — TARGET AND RELATED COLUMNS")

for col in [
    "churn_label",
    "churn_value",
    "churn_score",
    "churn_reason",
    "cltv"
]:
    if col in telecom_excel_ml.columns:
        print(f"\n{col}")
        print("Unique values:", telecom_excel_ml[col].nunique())
        print("Missing values:", telecom_excel_ml[col].isnull().sum())
        print(telecom_excel_ml[col].value_counts(dropna=False).head(10))

TELECOM EXCEL — TARGET AND RELATED COLUMNS

churn_label
Unique values: 2
Missing values: 0
churn_label
No     5174
Yes    1869
Name: count, dtype: int64

churn_value
Unique values: 2
Missing values: 0
churn_value
0    5174
1    1869
Name: count, dtype: int64

churn_score
Unique values: 85
Missing values: 0
churn_score
80    151
71    148
77    145
67    143
76    141
68    141
70    140
69    140
78    138
72    137
Name: count, dtype: int64

churn_reason
Unique values: 20
Missing values: 5174
churn_reason
NaN                                          5174
Attitude of support person                    192
Competitor offered higher download speeds     189
Competitor offered more data                  162
Don't know                                    154
Competitor made better offer                  140
Attitude of service provider                  135
Competitor had better devices                 130
Network reliability                           103
Product dissatisfaction               

In [18]:
print("BANKING — TARGET AND RELATED COLUMNS")

for col in [
    "exited",
    "complain",
    "satisfaction_score"
]:
    if col in banking_ml.columns:
        print(f"\n{col}")
        print("Unique values:", banking_ml[col].nunique())
        print("Missing values:", banking_ml[col].isnull().sum())
        print(banking_ml[col].value_counts(dropna=False).head(10))

if "complain" in banking_ml.columns:
    print("\nBanking churn rate by complain:")
    print(pd.crosstab(
        banking_ml["complain"],
        banking_ml["exited"],
        normalize="index"
    ).round(3))

BANKING — TARGET AND RELATED COLUMNS

exited
Unique values: 2
Missing values: 0
exited
0    7962
1    2038
Name: count, dtype: int64

complain
Unique values: 2
Missing values: 0
complain
0    7956
1    2044
Name: count, dtype: int64

satisfaction_score
Unique values: 5
Missing values: 0
satisfaction_score
3    2042
2    2014
4    2008
5    2004
1    1932
Name: count, dtype: int64

Banking churn rate by complain:
exited        0      1
complain              
0         0.999  0.001
1         0.005  0.995


In [19]:
excel_ids = set(
    telecom_excel_ml["customerid"].dropna().astype(str).str.strip()
)

csv_ids = set(
    telecom_csv_ml["customerid"].dropna().astype(str).str.strip()
)

overlapping_ids = excel_ids.intersection(csv_ids)

print("Excel unique customer IDs:", len(excel_ids))
print("CSV unique customer IDs:", len(csv_ids))
print("Overlapping customer IDs:", len(overlapping_ids))
print("Sample overlapping IDs:", list(overlapping_ids)[:10])

Excel unique customer IDs: 7043
CSV unique customer IDs: 7043
Overlapping customer IDs: 7043
Sample overlapping IDs: ['6051-PTVNS', '2080-CAZNM', '3519-ZKXGG', '2607-FBDFF', '0396-HUJBP', '4828-FAZPK', '0867-LDTTC', '6996-KNSML', '6179-GJPSO', '9167-APMXZ']


In [20]:
columns_to_check = [
    "churn_label",
    "churn_value",
    "churn_score",
    "churn_reason",
    "cltv"
]

for col in columns_to_check:
    if col in telecom_excel_ml.columns:
        print(f"\n--- {col} ---")
        print("Missing:", telecom_excel_ml[col].isnull().sum())
        print("Unique:", telecom_excel_ml[col].nunique())
        print(telecom_excel_ml[col].value_counts(dropna=False).head(10))


--- churn_label ---
Missing: 0
Unique: 2
churn_label
No     5174
Yes    1869
Name: count, dtype: int64

--- churn_value ---
Missing: 0
Unique: 2
churn_value
0    5174
1    1869
Name: count, dtype: int64

--- churn_score ---
Missing: 0
Unique: 85
churn_score
80    151
71    148
77    145
67    143
76    141
68    141
70    140
69    140
78    138
72    137
Name: count, dtype: int64

--- churn_reason ---
Missing: 5174
Unique: 20
churn_reason
NaN                                          5174
Attitude of support person                    192
Competitor offered higher download speeds     189
Competitor offered more data                  162
Don't know                                    154
Competitor made better offer                  140
Attitude of service provider                  135
Competitor had better devices                 130
Network reliability                           103
Product dissatisfaction                       102
Name: count, dtype: int64

--- cltv ---
Missing: 0
Uniq

In [21]:
excel_ids = set(
    telecom_excel_ml["customerid"].dropna().astype(str).str.strip()
)

csv_ids = set(
    telecom_csv_ml["customerid"].dropna().astype(str).str.strip()
)

overlapping_ids = excel_ids.intersection(csv_ids)

print("Excel unique IDs:", len(excel_ids))
print("CSV unique IDs:", len(csv_ids))
print("Overlapping IDs:", len(overlapping_ids))
print("Examples:", list(overlapping_ids)[:10])

Excel unique IDs: 7043
CSV unique IDs: 7043
Overlapping IDs: 7043
Examples: ['6051-PTVNS', '2080-CAZNM', '3519-ZKXGG', '2607-FBDFF', '0396-HUJBP', '4828-FAZPK', '0867-LDTTC', '6996-KNSML', '6179-GJPSO', '9167-APMXZ']


In [22]:
print("CLTV by churn outcome:")
display(
    telecom_excel_ml.groupby("churn_value")["cltv"]
    .agg(["count", "mean", "median", "min", "max"])
)

print("\nChurn score by churn outcome:")
display(
    telecom_excel_ml.groupby("churn_value")["churn_score"]
    .agg(["count", "mean", "median", "min", "max"])
)

CLTV by churn outcome:


,count,mean,median,min,max
churn_value,,,,,
0,5174,4490.921337,4620.0,2003,6500
1,1869,4149.414660,4238.0,2003,6484



Churn score by churn outcome:


,count,mean,median,min,max
churn_value,,,,,
0,5174,50.098183,50.0,5,80
1,1869,82.510433,82.0,65,100


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

In [23]:
TARGETS = {
    "Telecom Excel": "churn_value",
    "Telecom CSV": "churn",
    "Banking": "exited"
}

EXCLUDED_COLUMNS = {
    "Telecom Excel": [
        "customerid",
        "churn_label",
        "churn_score",
        "churn_reason",
        "cltv"
    ],
    "Telecom CSV": [
        "customerid"
    ],
    "Banking": [
        "customerid",
        "surname",
        "complain"
    ]
}

print("Target and exclusion rules defined.")

Target and exclusion rules defined.


In [24]:
datasets_ml = {
    "Telecom Excel": telecom_excel_ml,
    "Telecom CSV": telecom_csv_ml,
    "Banking": banking_ml
}

prepared_data = {}

for name, df in datasets_ml.items():

    target = TARGETS[name]
    excluded = EXCLUDED_COLUMNS[name]

    if target not in df.columns:
        raise KeyError(f"{target} not found in {name}")

    # Remove rows with missing target values
    df = df.dropna(subset=[target]).copy()

    # Separate target from input features
    X = df.drop(columns=[target] + excluded, errors="ignore")
    y = df[target].copy()

    # Remove columns that are entirely empty
    empty_columns = X.columns[X.isnull().all()].tolist()
    X = X.drop(columns=empty_columns)

    prepared_data[name] = {
        "X": X,
        "y": y
    }

    print("\n", name)
    print("Features:", X.shape)
    print("Target:", y.shape)
    print("Excluded empty columns:", empty_columns)
    print("Target distribution:")
    print(y.value_counts())


 Telecom Excel
Features: (7043, 27)
Target: (7043,)
Excluded empty columns: []
Target distribution:
churn_value
0    5174
1    1869
Name: count, dtype: int64

 Telecom CSV
Features: (7043, 19)
Target: (7043,)
Excluded empty columns: []
Target distribution:
churn
No     5174
Yes    1869
Name: count, dtype: int64

 Banking
Features: (10000, 13)
Target: (10000,)
Excluded empty columns: []
Target distribution:
exited
0    7962
1    2038
Name: count, dtype: int64


In [25]:
split_data = {}

for name, data in prepared_data.items():

    X = data["X"]
    y = data["y"]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )

    split_data[name] = {
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test
    }

    print("\n" + "=" * 50)
    print(name)
    print("=" * 50)

    print("Training features:", X_train.shape)
    print("Testing features:", X_test.shape)

    print("\nTraining target distribution:")
    print(y_train.value_counts(normalize=True).round(3))

    print("\nTesting target distribution:")
    print(y_test.value_counts(normalize=True).round(3))

NameError: name 'train_test_split' is not defined